# BO coherence / séries
Vérifier la cohérence du bo_type (1/2/3/5), des scores finaux et du nombre de maps.

In [2]:
import sys
from pathlib import Path
import polars as pl

def _find_root():
    cand = Path.cwd()
    for c in [cand, *cand.parents]:
        if (c / "src").exists() and (c / "data").exists():
            return c
    return cand

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.dota_data import read_processed_tables

tables = read_processed_tables(ROOT / "data" / "processed")
matches = tables["matches"]
series_path = ROOT / "data" / "processed" / "series.parquet"
series_df = pl.read_parquet(series_path) if series_path.exists() else None
series_maps_path = ROOT / "data" / "metrics" / "series_maps.parquet"
series_maps = pl.read_parquet(series_maps_path) if series_maps_path.exists() else None

TARGET_LEAGUE = None  # ex: 16477
TARGET_SERIES = None  # ex: 863552

BO_ALLOWED = {1: {1}, 2: {1, 2}, 3: {2, 3}, 5: {3, 4, 5}}

def maps_ok(bo, count):
    return count in BO_ALLOWED.get(bo, set())


## Distribution des BO et scores

In [3]:
if series_df is None or series_df.is_empty():
    print("series.parquet manquant ou vide (relancer make parquet)")
else:
    dist_bo = series_df.group_by("bo_type").agg(pl.len().alias("series")).sort("bo_type")
    display(dist_bo)

    scores = series_df.select(
        "bo_type",
        "match_count",
        "score_team_a",
        "score_team_b",
        "series_id",
        "leagueid",
    )
    display(scores.head())


bo_type,series
i64,u32
1,1072
2,852
3,4471
5,199


bo_type,match_count,score_team_a,score_team_b,series_id,leagueid
i64,i64,i64,i64,i64,i64
3,3,2,1,1039062,18920
3,3,1,2,1038719,18920
3,3,2,1,1038288,18920
3,2,2,0,1037873,18920
3,2,0,2,1036811,18920


## Anomalies : map_count vs bo_type et équipes

In [4]:
if series_df is None or series_df.is_empty():
    print("series.parquet manquant")
else:
    anomalies = (
        series_df
        .with_columns(pl.struct(["bo_type","match_count"]).map_elements(lambda s: maps_ok(s["bo_type"], s["match_count"])).alias("maps_ok"))
        .with_columns((pl.col("teams_in_series") != 2).alias("bad_teams"))
        .filter((~pl.col("maps_ok")) | pl.col("bad_teams"))
        .sort(["bo_type","match_count","series_id","leagueid"])
    )
    display(anomalies)


/tmp/ipykernel_126164/649738606.py:6: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  .with_columns(pl.struct(["bo_type","match_count"]).map_elements(lambda s: maps_ok(s["bo_type"], s["match_count"])).alias("maps_ok"))


series_id,leagueid,league_name,tournament_name,tournament_slug,tournament_tier,tournament_location,series_type_raw,bo_type,match_count,teams,teams_in_series,team_a,team_b,score_team_a,score_team_b,winner_team_id,max_wins,start_time_min,start_time_max,maps_ok,bad_teams
i64,i64,str,str,str,str,str,i64,i64,i64,list[i64],i64,i64,i64,i64,i64,i64,i64,i64,i64,bool,bool


## Séries par ligue (synthèse anomalies)

In [5]:
if series_df is None or series_df.is_empty():
    print("series.parquet manquant")
else:
    by_league = (
        series_df
        .with_columns(pl.struct(["bo_type","match_count"]).map_elements(lambda s: maps_ok(s["bo_type"], s["match_count"])).alias("maps_ok"))
        .with_columns((pl.col("teams_in_series") != 2).alias("bad_teams"))
        .group_by(["leagueid","league_name","tournament_name"])
        .agg(
            pl.len().alias("series"),
            pl.sum((~pl.col("maps_ok")) | pl.col("bad_teams")).alias("anomaly_series"),
        )
        .sort("anomaly_series", descending=True)
    )
    display(by_league)


/tmp/ipykernel_126164/1882310172.py:6: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  .with_columns(pl.struct(["bo_type","match_count"]).map_elements(lambda s: maps_ok(s["bo_type"], s["match_count"])).alias("maps_ok"))


TypeError: invalid input for `col`

Expected `str` or `DataType`, got 'Expr'.

## Focus filters (league/series) + maps

In [ ]:
if series_df is None or series_df.is_empty():
    print("series.parquet manquant")
else:
    filt = series_df
    if TARGET_LEAGUE is not None:
        filt = filt.filter(pl.col("leagueid") == TARGET_LEAGUE)
    if TARGET_SERIES is not None:
        filt = filt.filter(pl.col("series_id") == TARGET_SERIES)
    if filt.is_empty():
        print("Aucune série pour ces filtres")
    else:
        display(filt.sort(["series_id","bo_type"]))
        if series_maps is not None:
            sm = series_maps
            if TARGET_LEAGUE is not None:
                sm = sm.filter(pl.col("leagueid") == TARGET_LEAGUE)
            if TARGET_SERIES is not None:
                sm = sm.filter(pl.col("series_id") == TARGET_SERIES)
            display(sm.sort(["series_id","map_num","start_time"]))
        else:
            print("series_maps.parquet manquant")


series_id,leagueid,league_name,tournament_name,tournament_slug,tournament_tier,tournament_location,series_type_raw,bo_type,match_count,teams,teams_in_series,team_a,team_b,score_team_a,score_team_b,winner_team_id,max_wins,start_time_min,start_time_max
i64,i64,str,str,str,str,str,i64,i64,i64,list[i64],i64,i64,i64,i64,i64,i64,i64,i64,i64
0,18830,"""BLAST Slam V: …","""BLAST Slam V: …","""blast-slam-v-c…","""regular""","""online""",0,1,1,"[9444076, 9315393]",2,9444076,9315393,0,1,9315393,1,1761051142,1761051142
816359,16846,"""FISSURE Univer…","""FISSURE Univer…","""fissure-univer…","""regular""","""online""",1,3,3,"[8728920, 8291895]",2,8728920,8291895,1,2,8291895,2,1724355059,1724363401
838795,16093,"""BetBoom Dacha …","""BetBoom Dacha …","""betboom-dacha-…","""regular""","""online""",1,1,1,"[9081007, 8588969]",2,9081007,8588969,1,0,9081007,1,1704355281,1704355281
838796,16093,"""BetBoom Dacha …","""BetBoom Dacha …","""betboom-dacha-…","""regular""","""online""",1,3,2,"[2576071, 9279613]",2,2576071,9279613,2,0,2576071,2,1704355380,1704358994
838802,16093,"""BetBoom Dacha …","""BetBoom Dacha …","""betboom-dacha-…","""regular""","""online""",1,1,1,"[9081007, 8588969]",2,9081007,8588969,1,0,9081007,1,1704358792,1704358792
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1044594,18866,"""European Pro L…","""European Pro L…","""european-pro-l…","""regular""","""online""",1,3,2,"[9017006, 9303383]",2,9017006,9303383,0,2,9303383,2,1765198839,1765202381
1044694,18866,"""European Pro L…","""European Pro L…","""european-pro-l…","""regular""","""online""",1,3,2,"[9886449, 2576071]",2,9886449,2576071,0,2,2576071,2,1765217016,1765220615
1044854,19054,"""Snake Trophy""","""Snake Trophy""","""snake-trophy""","""regular""","""online""",1,3,3,"[9546449, 9722899]",2,9546449,9722899,2,1,9546449,2,1765256683,1765263782


series_id,leagueid,series_type,bo_type,map_num,match_id,start_time,radiant_team_id,dire_team_id,radiant_win
i64,i64,i64,i64,u32,i64,i64,i64,i64,bool
0,18830,1,1,1,8521719394,1761051142,9444076,9315393,false
816359,16846,3,3,1,7909186596,1724355059,8728920,8291895,true
816359,16846,3,3,2,7909275659,1724358805,8291895,8728920,true
816359,16846,3,3,3,7909343508,1724363401,8728920,8291895,false
838795,16093,1,1,1,7520706781,1704355281,9081007,8588969,true
…,…,…,…,…,…,…,…,…,…
1044854,19054,3,3,3,8597003724,1765263782,9546449,9722899,true
1044898,18866,3,3,1,8597139842,1765274867,9895247,9600141,true
1044898,18866,3,3,2,8597191886,1765278054,9895247,9600141,true
